<a href="https://colab.research.google.com/github/maivu18082005-hub/Parameter_Identification_of_PMSM_using_PINN/blob/main/04_train_pinn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ===== CELL 1: KẾT NỐI DRIVE & KIỂM TRA GPU =====
from google.colab import drive  # Import thư viện drive của Colab để gắn kết nối tới Google Drive
import torch                    # Import thư viện PyTorch làm framework chính cho Deep Learning
import os                       # Import thư viện os để thực hiện các thao tác kiểm tra thư mục và đường dẫn

drive.mount('/content/drive')   # Thực hiện Mount (gắn) Google Drive của bạn vào môi trường ảo Colab

# Khai báo đường dẫn thư mục dự án gốc trên Google Drive
DRIVE_PROJECT = '/content/drive/MyDrive/Parameter_Identification_of_PMSM_using_PINN'

# Khai báo đường dẫn chứa tập dữ liệu đã qua tiền xử lý (.npy và .pkl)
PROCESSED_DATA_DIR = os.path.join(DRIVE_PROJECT, 'dataset', 'processed')

# Khai báo thư mục lưu trữ trọng số mô hình tốt nhất (Checkpoints)
CHECKPOINT_DIR = os.path.join(DRIVE_PROJECT, 'checkpoints')

# Tự động tạo thư mục checkpoints trên Google Drive nếu thư mục này chưa tồn tại
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Kiểm tra xem Colab có GPU không: nếu có thì dùng 'cuda', nếu không thì dùng 'cpu'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Thiết bị tính toán đang dùng:', device) # In ra màn hình thiết bị đang chạy (ví dụ: cuda:0)

Mounted at /content/drive
Thiết bị tính toán đang dùng: cuda


In [2]:
# ===== CELL 2: PYTORCH DATASET CHO PINN (TỐI ƯU TỐC ĐỘ) =====
from torch.utils.data import Dataset, DataLoader  # Import công cụ quản lý và nạp dữ liệu theo Batch
import numpy as np                                # Import NumPy để đọc mảng .npy

class PMSMPINNDataset(Dataset):
    """
    Lớp Dataset nạp Input (X), Target (Y) và cờ trạng thái xác lập (Steady)
    để phục vụ tính toán hàm Loss vật lý chính xác cho mạng PINN.
    """
    def __init__(self, data_dir, split='train'):
        # Nạp Input, Target và cờ Steady Mask từ thư mục processed và chuyển sang Tensor
        self.X = torch.tensor(np.load(os.path.join(data_dir, f'X_{split}.npy')), dtype=torch.float32)
        self.Y = torch.tensor(np.load(os.path.join(data_dir, f'Y_{split}.npy')), dtype=torch.float32)
        self.steady = torch.tensor(np.load(os.path.join(data_dir, f'Steady_{split}.npy')), dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        # Trả về bộ 3 giá trị: Input, Target và cờ đánh dấu xác lập (1.0 hoặc 0.0)
        return self.X[idx], self.Y[idx], self.steady[idx]

# Khởi tạo 3 tập dữ liệu Train, Validation và Test
train_dataset = PMSMPINNDataset(PROCESSED_DATA_DIR, split='train')
val_dataset = PMSMPINNDataset(PROCESSED_DATA_DIR, split='validation')
test_dataset = PMSMPINNDataset(PROCESSED_DATA_DIR, split='test')

# Khởi tạo DataLoader tối ưu cho GPU T4 với batch_size lớn và pin_memory
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=1024, shuffle=False, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False, pin_memory=True)

print(f'Đã nạp Dataset PINN! Train: {len(train_dataset)} mẫu.')

Đã nạp Dataset PINN! Train: 900000 mẫu.


In [3]:
# ===== CELL 3: KIẾN TRÚC MẠNG PINN & PHYSICS LOSS (ĐÃ SỬA PSI_M & LỌC QUÁ ĐỘ) =====
import torch.nn as nn  # Import module mạng Neural của PyTorch
import joblib          # Import joblib để đọc bộ chuẩn hóa Scaler

# Nạp lại X_scaler và Y_scaler để giải chuẩn hóa dữ liệu về đơn vị vật lý (V, A, rpm, Ohm, H)
X_scaler = joblib.load(os.path.join(PROCESSED_DATA_DIR, 'X_scaler.pkl'))
Y_scaler = joblib.load(os.path.join(PROCESSED_DATA_DIR, 'Y_scaler.pkl'))

# Đưa giá trị trung bình (mean) và độ lệch chuẩn (scale) lên GPU dưới dạng Tensor
x_mean = torch.tensor(X_scaler.mean_, dtype=torch.float32).to(device)
x_scale = torch.tensor(X_scaler.scale_, dtype=torch.float32).to(device)
y_mean = torch.tensor(Y_scaler.mean_, dtype=torch.float32).to(device)
y_scale = torch.tensor(Y_scaler.scale_, dtype=torch.float32).to(device)

class PMSM_PINN(nn.Module):
    """
    Kiến trúc mạng PINN gồm 5 đầu vào [ud, uq, Id, Iq, nm], 3 lớp ẩn 128 nút (Tanh)
    và 3 đầu ra ước lượng tham số [Rs, Ld, Lq].
    """
    def __init__(self, input_dim=5, output_dim=3, hidden_dim=128):
        super(PMSM_PINN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.network(x)

def compute_physics_loss(inputs_scaled, outputs_scaled, steady_mask, pole_pairs=4, psi_m=0.527139):
    """
    Hàm tính sai số vật lý (Physics-informed Loss) dựa trên phương trình điện áp trạng thái xác lập của PMSM.
    Chỉ tính trên các mẫu thỏa mãn điều kiện xác lập (steady_mask == 1.0) và dùng đúng psi_m = 0.527139 Wb.
    """
    # Tạo mặt nạ lọc ra các mẫu đang ở trạng thái xác lập trong Batch
    mask = steady_mask > 0.5
    # Nếu cả Batch không có mẫu xác lập nào thì trả về loss vật lý bằng 0
    if mask.sum() == 0:
        return torch.tensor(0.0, device=inputs_scaled.device)

    # Chỉ giữ lại dữ liệu của các mẫu xác lập
    in_s = inputs_scaled[mask]
    out_s = outputs_scaled[mask]

    # Giải chuẩn hóa (Inverse StandardScaler): Giá_trị_thực = Giá_trị_chuẩn_hóa * Std + Mean
    inputs_real = in_s * x_scale + x_mean
    outputs_real = out_s * y_scale + y_mean

    # Tách các biến điện áp, dòng điện và tốc độ thực tế
    ud = inputs_real[:, 0]  # Điện áp trục d (V)
    uq = inputs_real[:, 1]  # Điện áp trục q (V)
    Id = inputs_real[:, 2]  # Dòng điện trục d (A)
    Iq = inputs_real[:, 3]  # Dòng điện trục q (A)
    nm = inputs_real[:, 4]  # Tốc độ cơ học (rpm)

    # Tách 3 tham số do mạng PINN dự đoán
    Rs_est = outputs_real[:, 0]  # Điện trở stator ước lượng (Ohm)
    Ld_est = outputs_real[:, 1]  # Điện cảm trục d ước lượng (H)
    Lq_est = outputs_real[:, 2]  # Điện cảm trục q ước lượng (H)

    # Quy đổi tốc độ cơ học nm (vòng/phút) sang tốc độ góc điện omega_e (rad/s)
    omega_m = nm * (2.0 * np.pi / 60.0)
    omega_e = pole_pairs * omega_m

    # Tính điện áp ud, uq lý thuyết theo phương trình trạng thái xác lập động cơ PMSM
    ud_calc = Rs_est * Id - omega_e * Lq_est * Iq
    uq_calc = Rs_est * Iq + omega_e * (Ld_est * Id + psi_m)

    # Tính sai số bình phương trung bình (MSE) giữa điện áp mô phỏng và điện áp vật lý tính được
    loss_ud = torch.mean((ud - ud_calc) ** 2)
    loss_uq = torch.mean((uq - uq_calc) ** 2)

    return loss_ud + loss_uq

# Khởi tạo mạng PINN và chuyển lên GPU
model_pinn = PMSM_PINN().to(device)
print("Đã khởi tạo mô hình PINN với psi_m = 0.527139 Wb trên thiết bị:", device)

Đã khởi tạo mô hình PINN với psi_m = 0.527139 Wb trên thiết bị: cuda


In [4]:
# ===== CELL 4: HUẤN LUYỆN MÔ HÌNH PINN (LƯU TẠM LOCAL ĐỂ TRÁNH NGHẼN DRIVE I/O) =====
import torch.optim as optim  # Import bộ tối ưu Optimizer
import shutil                # Import thư viện copy file
import time                  # Import thư viện đo thời gian

criterion_data = nn.MSELoss()                               # Hàm Loss dữ liệu (MSE)
optimizer = optim.Adam(model_pinn.parameters(), lr=0.001)   # Bộ tối ưu Adam với lr = 0.001
# Tự động giảm một nửa learning rate nếu Val Loss đi ngang sau 10 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

lambda_p = 1e-3  # Hệ số trọng số cân bằng thành phần Physics Loss trong tổng hàm Loss
epochs = 100     # Tổng số epochs huấn luyện
best_val_loss = float('inf')

# Đường dẫn lưu tạm trên ổ cứng Colab và đường dẫn lưu chính thức trên Google Drive
LOCAL_CKPT = '/content/best_pinn_model.pth'
DRIVE_CKPT = os.path.join(CHECKPOINT_DIR, 'best_pinn_model.pth')

print("=== BẮT ĐẦU HUẤN LUYỆN PINN ===")
start_time = time.time()

for epoch in range(1, epochs + 1):
    # --- GIAI ĐOẠN 1: HUẤN LUYỆN (TRAIN) ---
    model_pinn.train()
    train_data_loss = 0.0
    train_phys_loss = 0.0

    for inputs, targets, steady in train_loader:
        # Đẩy dữ liệu Batch lên GPU
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        steady = steady.to(device, non_blocking=True)

        optimizer.zero_grad()           # Đặt lại gradient về 0
        outputs = model_pinn(inputs)    # Dự đoán tham số [Rs, Ld, Lq]

        loss_d = criterion_data(outputs, targets)                  # Tính sai số dữ liệu (Data Loss)
        loss_p = compute_physics_loss(inputs, outputs, steady)     # Tính sai số vật lý (Physics Loss)
        loss_total = loss_d + lambda_p * loss_p                    # Tổng hợp hàm Loss của PINN

        loss_total.backward()           # Lan truyền ngược tính đạo hàm
        optimizer.step()                # Cập nhật trọng số mạng

        # Cộng dồn giá trị Loss để thống kê cuối mỗi Epoch
        train_data_loss += loss_d.item() * inputs.size(0)
        train_phys_loss += loss_p.item() * inputs.size(0)

    train_data_loss /= len(train_dataset)
    train_phys_loss /= len(train_dataset)

    # --- GIAI ĐOẠN 2: ĐÁNH GIÁ (VALIDATION) ---
    model_pinn.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, targets, _ in val_loader:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            outputs = model_pinn(inputs)
            val_loss += criterion_data(outputs, targets).item() * inputs.size(0)

    val_loss /= len(val_dataset)
    scheduler.step(val_loss) # Điều chỉnh Learning Rate nếu cần

    # Lưu trọng số tốt nhất vào bộ nhớ cục bộ /content
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model_pinn.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss
        }, LOCAL_CKPT)

    # In tiến độ huấn luyện và thời gian chạy sau mỗi 10 epochs
    if epoch % 10 == 0 or epoch == 1:
        elapsed = (time.time() - start_time) / 60.0
        print(f"Epoch [{epoch:3d}/{epochs}] | Data Loss: {train_data_loss:.6f} | Phys Loss: {train_phys_loss:.4f} | Val Loss: {val_loss:.6f} | Thời gian: {elapsed:.2f} phút")

# Copy file trọng số tốt nhất từ Colab sang Google Drive sau khi train hoàn tất
shutil.copy(LOCAL_CKPT, DRIVE_CKPT)
print(f"\n--> Hoàn thành trong {(time.time() - start_time)/60:.2f} phút! Đã lưu vào: {DRIVE_CKPT}")

=== BẮT ĐẦU HUẤN LUYỆN PINN ===
Epoch [  1/100] | Data Loss: 0.262663 | Phys Loss: 269.0674 | Val Loss: 0.592114 | Thời gian: 0.29 phút
Epoch [ 10/100] | Data Loss: 0.101829 | Phys Loss: 269.2934 | Val Loss: 0.612676 | Thời gian: 2.73 phút
Epoch [ 20/100] | Data Loss: 0.068963 | Phys Loss: 269.2872 | Val Loss: 0.631110 | Thời gian: 5.45 phút
Epoch [ 30/100] | Data Loss: 0.062645 | Phys Loss: 269.2853 | Val Loss: 0.651543 | Thời gian: 8.14 phút
Epoch [ 40/100] | Data Loss: 0.059328 | Phys Loss: 269.2867 | Val Loss: 0.593720 | Thời gian: 10.87 phút
Epoch [ 50/100] | Data Loss: 0.057070 | Phys Loss: 269.2851 | Val Loss: 0.580010 | Thời gian: 13.58 phút
Epoch [ 60/100] | Data Loss: 0.053388 | Phys Loss: 269.2901 | Val Loss: 0.543176 | Thời gian: 16.33 phút
Epoch [ 70/100] | Data Loss: 0.051145 | Phys Loss: 269.2888 | Val Loss: 0.528904 | Thời gian: 19.08 phút
Epoch [ 80/100] | Data Loss: 0.050405 | Phys Loss: 269.2875 | Val Loss: 0.547589 | Thời gian: 21.79 phút
Epoch [ 90/100] | Data Loss

In [5]:
# ===== CELL 5: ĐÁNH GIÁ PINN TRÊN TẬP TEST =====

# Nạp bản trọng số tốt nhất đã lưu trên Google Drive ra để đánh giá
checkpoint = torch.load(os.path.join(CHECKPOINT_DIR, 'best_pinn_model.pth'))
model_pinn.load_state_dict(checkpoint['model_state_dict']) # Khôi phục trạng thái trọng số
model_pinn.eval() # Chuyển mô hình sang chế độ dự đoán/đánh giá

all_preds = []   # Mảng chứa toàn bộ kết quả dự đoán từ mạng PINN
all_targets = [] # Mảng chứa toàn bộ nhãn thực tế từ mô phỏng

with torch.no_grad(): # Tắt tính toán gradient khi kiểm tra tập Test
    for inputs, targets, _ in test_loader:
        inputs = inputs.to(device)                     # Đẩy dữ liệu lên GPU
        outputs = model_pinn(inputs)                  # Mạng PINN dự đoán
        all_preds.append(outputs.cpu().numpy())       # Lưu dự đoán về CPU dưới dạng NumPy
        all_targets.append(targets.numpy())           # Lưu nhãn thực tế dưới dạng NumPy

# Ghép các Batch nhỏ thành mảng 2D hoàn chỉnh
all_preds = np.vstack(all_preds)
all_targets = np.vstack(all_targets)

# Giải chuẩn hóa (Inverse Transform) kết quả dự đoán và nhãn thực tế về đơn vị thực (Ohm và Henry)
preds_real = Y_scaler.inverse_transform(all_preds)
targets_real = Y_scaler.inverse_transform(all_targets)

# Tính sai số phần trăm tuyệt đối trung bình (MAPE) cho từng tham số: Rs, Ld, Lq
mape = np.mean(np.abs((targets_real - preds_real) / targets_real), axis=0) * 100

print("=== KẾT QUẢ ĐÁNH GIÁ PINN TRÊN TẬP TEST ===")
print(f"Sai số Rs trung bình (PINN): {mape[0]:.2f}%") # In phần trăm sai số của Rs
print(f"Sai số Ld trung bình (PINN): {mape[1]:.2f}%") # In phần trăm sai số của Ld
print(f"Sai số Lq trung bình (PINN): {mape[2]:.2f}%") # In phần trăm sai số của Lq

=== KẾT QUẢ ĐÁNH GIÁ PINN TRÊN TẬP TEST ===
Sai số Rs trung bình (PINN): 6.01%
Sai số Ld trung bình (PINN): 0.12%
Sai số Lq trung bình (PINN): 0.89%
